In [0]:
# Solución Parcial 2 : Análisis de datos del mercado laboral GEIH 2026
# Presentado por: Oscar Hernado Agudelo. Código: 9.728.931

In [0]:
# Nota: El desarrollo de esta actividad contó con la ayuda de la IA, los comentarios son creación propia. 

In [0]:
import pandas as pd
import numpy as np

#Leyendo la tabla de caracteristicas generales, desempleados, ft y ocupados
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
ft = spark.read.format("delta").table("workspace.electiva.ft")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")

# Convertir el DataFrame de Spark a Pandas
cgse = cgse.toPandas()
desempleados=desempleados.toPandas()
ft=ft.toPandas()
ocupados=ocupados.toPandas()

In [0]:
# Solución del punto uno del parcial

In [0]:
# Partimos con la tabla de Caracteristicas generales, seguridad social en salud y educacion (cgse) que contiene la información general de cada colombiano, sin embargo, para el proposito de resolver los puntos solicitados, necesitamos información adicional del mercado laboral. Usamos como llaves del cgse los parametros 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN', que nos permite encajar las columnas de las otras tablas correctamente en la tabla cgse. Para ello creamos una variable llamada llaves con la lista de 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN' para no tener que escribir esta lista cada vez que se haga uso del comando pd.merge().

llaves = ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN']

# Con el comando pd.merge() introducimos en la tabla cgse las columnas PET y FT de la tabla ft. Al introducir esta columna tenemos acceso a los datos de la población en edad de trabajar y la económicamente activa

cgse = pd.merge(cgse, ft[llaves + ['PET', 'FT']], on=llaves, how='left')

# Ahora introducimos en la tabla cgse las columnas DSI de desocupados de la tabla desempleados

cgse = pd.merge(cgse, desempleados[llaves + ['DSI']], on=llaves, how='left')

# Ahora introducimos en la tabla cgse las columnas de la población ocupada y la de ingresos laborales (OCI,INGLABO) de la tabla ocupados

cgse = pd.merge(cgse, ocupados[llaves + ['OCI', 'INGLABO']], on=llaves, how='left')

# Como habran valores vacíos en las columnas DSI y OCI de la tabla de ocupados y desempleados, reemplazamos estos valores por 0, estos nos permitirá realizar operaciones matemáticas con estas columnas

cgse['DSI'] = cgse['DSI'].fillna(0)
cgse['OCI'] = cgse['OCI'].fillna(0)



In [0]:
# Calculo de la población en edad de trabajar a partir del total nacional. Se toma de la tabla cgse la variable P6040 que contiene la información de la edad de la población; se procede a encontrar aquellos que tengan quince o más años y se le aplica el factor de expanción. El resultado de la población en edad de trabajar a junio de 2026 es 41.202.631 personas en Colombia

PET = cgse[cgse['P6040'] >= 15]['FEX_C18'].sum()
print(f"Población en Edad de Trabajar (PET): {PET:,.0f} personas.")

In [0]:
# La población en edad de trabajar en Colombia a junio de 2026 según GEIH - 2026 es de 41.202.631 personas.

In [0]:
# El calculo de la población que NO está en edad de trabajar se realiza de la misma manera que el cálculo de la población en edad de trabajar, solo que comparamos la variable P6040 con aquellas personas que tienen menos de quince años.

NO_PET = cgse[cgse['P6040'] < 15]['FEX_C18'].sum()
print(f"Población que NO está en Edad de Trabajar: {NO_PET:,.0f} personas.")

In [0]:
# La población que NO está en edad de trabajar en Colombia a junio de 2026 según GEIH - 2026 es de 11.346.181 personas.

In [0]:
# Cálculo de la Población Económicamente Activa (PEA) a nivel nacional. Se toman de la tabla cgse las personas vinculadas a la fuerza de trabajo (FT == 1) y se realiza la sumatoria de su factor de expansión (FEX_C18).

PEA = cgse[cgse['FT'] == 1]['FEX_C18'].sum()
print(f"Población Económicamente Activa (PEA): {PEA:,.0f} personas.")

In [0]:
# La población Económicamente Activa (PEA) en Colombia a junio de 2026 según GEIH - 2026 es de 26.593.429 personas.

In [0]:
# Cálculo de la Población Económicamente Inactiva (PEI) a nivel nacional. Corresponde a la Población Fuera de la Fuerza de Trabajo (PFFT), obtenida al restar la fuerza de trabajo activa (PEA) del total de la población en edad de trabajar (PET).

PEI = PET - PEA
print(f"Población Económicamente Inactiva (PEI): {PEI:,.0f} personas.")

In [0]:
# La población Económicamente Inactiva (PEI) en Colombia a junio de 2026 según GEIH - 2026 es de 14.609.202 personas.

In [0]:
# Para hallar la tasa de ocupación y desocupación calculamos primero la cantidad total de personas ocupadas a nivel nacional que se encuentra en la columna OCI, el comparador lógico con 1 nos garantiza que sea afirmativa. Aplicamos luego la fórmula del DANE (Ocupados / PET) * 100 y (Desocupados / PEA) * 100. Hay que tener presente que Desocupados se puede hallar como (PEA - Ocupados)

Ocupados = cgse[cgse['OCI'] == 1]['FEX_C18'].sum()
tasa_ocupacion = (Ocupados / PET) * 100
tasa_desempleo = ((PEA - Ocupados) / PEA) * 100

print(f"• Tasa de Ocupación (TO): {tasa_ocupacion:.2f}%")
print(f"• Tasa de Desempleo (TD): {tasa_desempleo:.2f}%")

In [0]:
# La tasa de ocupación (TO) en Colombia a junio de 2026 según GEIH - 2026 es de 59.36%
# La tasa de desempleo (TD) en Colombia a junio de 2026 según GEIH - 2026 es de 8.03%

In [0]:
# Solución del punto dos del parcial

In [0]:
# 1. Como debemos calcular la tasa de desempleo de cada departamento accedemos a la columna DSI y aplicamos la funcion np.where() que es un comando lógico que identifica si la persona está desocupada (1) o no (0), despues de ello se aplica el factor de expansión

cgse['Desocupados_Exp'] = np.where(cgse['DSI'] == 1, 1, 0) * cgse['FEX_C18']
cgse['PEA_Exp'] = np.where(cgse['FT'] == 1, 1, 0) * cgse['FEX_C18']

# 2. Se crea un directorio de departamentos (dic_dptos) para mapear nombres de departamentos basado en los códigos asignados por el DANE para cada uno

dic_dptos = {
    '05': 'Antioquia', '08': 'Atlántico', '11': 'Bogotá D.C.', '13': 'Bolívar', '15': 'Boyacá',
    '17': 'Caldas', '18': 'Caquetá', '19': 'Cauca', '20': 'Cesar', '23': 'Córdoba',
    '25': 'Cundinamarca', '27': 'Chocó', '41': 'Huila', '44': 'La Guajira', '47': 'Magdalena',
    '50': 'Meta', '52': 'Nariño', '54': 'Norte de Santander', '63': 'Quindío', '66': 'Risaralda',
    '68': 'Santander', '70': 'Sucre', '73': 'Tolima', '76': 'Valle del Cauca', '81': 'Arauca',
    '85': 'Casanare', '86': 'Putumayo', '88': 'San Andrés', '91': 'Amazonas', '94': 'Guainía',
    '95': 'Guaviare', '97': 'Vaupés', '99': 'Vichada'
}

# Hacemos ahora uso de la función groupby() que toma la tabla cgse ya modificada con las columnas adicionadas de las otras tablas y busca en la columna DPTO los codigos asignados en cada departamento para agruparlos en una única columna y especifica que solo interesa los valores de las columnas Desocupados_Exp y PEA_Exp. Una vez realizado este proceso crea una columna nueva llamada TD_General que aplica la fórmula de la tasa de desempleo (Desocupados / PEA)

dpto_general = cgse.groupby('DPTO')[['Desocupados_Exp', 'PEA_Exp']].sum().reset_index()
dpto_general['TD_General'] = (dpto_general['Desocupados_Exp'] / dpto_general['PEA_Exp']) * 100

# Homologar el código a 2 caracteres de texto para cruzar con el diccionario
dpto_general['DPTO'] = dpto_general['DPTO'].astype(float).astype(int).astype(str).str.zfill(2)
dpto_general['Nombre'] = dpto_general['DPTO'].map(dic_dptos)

# Procesdemos ahora a identificar los valores extremos del mercado laboral a nivel departamental. Para ello usamos la función  .idxmax() e .idxmin() para encontrar las filas con la mayor y menor tasa de desempleo respectivamente, y mediante la función .loc[] extraemos la información completa de esos dos departamentos.

extremo_mayor = dpto_general.loc[dpto_general['TD_General'].idxmax()]
extremo_menor = dpto_general.loc[dpto_general['TD_General'].idxmin()]

print("================================================================")
print("             ANÁLISIS DE EXTREMOS REGIONALES                    ")
print("================================================================")
print(f"• MAYOR DESEMPLEO: {extremo_mayor['Nombre']} (Código {extremo_mayor['DPTO']}) con {extremo_mayor['TD_General']:.2f}%")
print(f"• MENOR DESEMPLEO: {extremo_menor['Nombre']} (Código {extremo_menor['DPTO']}) con {extremo_menor['TD_General']:.2f}%")
print("================================================================\n")

# Vamos agrupar las variables 'DPTO' (Departamento) y 'P3271' (Sexo), separando las mujeres de los hombres, al segmentarlos podemos sumarlos y usar las variables Desocupados_Exp' y 'PEA_Exp para determinar el total de desempleados de hombres y mujeres por departamento.

dpto_sexo = cgse.groupby(['DPTO', 'P3271'])[['Desocupados_Exp', 'PEA_Exp']].sum().reset_index()

# Calculamos la tasa de desempleo relativa para sexo

dpto_sexo['Tasa_Desempleo'] = (dpto_sexo['Desocupados_Exp'] / dpto_sexo['PEA_Exp']) * 100

# A la columna DPTO aplicamos las siguientes transformaciones: la convertimos consecutivamente a decimal, entero y texto, aplicando .zfill(2) para asegurar que todos los códigos tengan dos dígitos con cero a la izquierda. Ademas, eemplazamos los códigos de departamento por sus nombres reales usando el diccionario 'dic_dptos'. Traducimos los códigos numéricos de la variable de sexo (P3271) a etiquetas de texto ('Hombre' / 'Mujer') para facilitar la lectura del reporte final.

dpto_sexo['DPTO'] = dpto_sexo['DPTO'].astype(float).astype(int).astype(str).str.zfill(2)
dpto_sexo['Nombre_Departamento'] = dpto_sexo['DPTO'].map(dic_dptos)
dpto_sexo['Sexo_Texto'] = dpto_sexo['P3271'].map({1: 'Hombre', 2: 'Mujer'})

# Creamos una nueva tabla independiente utilizando el método .copy() para almacenar únicamente las columnas que se desean mostrar (Código, Nombre del Departamento, Sexo y la Tasa de Desempleo calculada).

tabla_estructurada_p2 = dpto_sexo[['DPTO', 'Nombre_Departamento', 'Sexo_Texto', 'Tasa_Desempleo']].copy()

# Renombramos las columnas 

tabla_estructurada_p2.columns = ['Código DANE', 'Nombre del Departamento', 'Sexo', 'Tasa de Desempleo (%)']

# Redondear el indicador a dos decimales 

tabla_estructurada_p2['Tasa de Desempleo (%)'] = tabla_estructurada_p2['Tasa de Desempleo (%)'].round(2)

print("================================================================")
print("     TABLA FINAL: INDICADORES POR DEPARTAMENTO Y GÉNERO         ")
print("================================================================")
print(tabla_estructurada_p2.to_string(index=False))
print("================================================================")

In [0]:
# la mayor tasa de desempleo general en el país es Putumayo (22.18%), lo cual refleja la vulnerabilidad de su economía local. En contraste, el departamento con la menor tasa de desempleo es Vaupés (2.46%), un comportamiento asociado metodológicamente a una alta absorción en la informalidad y actividades de subsistencia rústica.

In [0]:
# Solución del punto tres del parcial

In [0]:
# Primero definimos una función condicional para clasificar la edad en los rangos de interes. Para ello hacemos uso de la función if que en el caso de que la condición se cumpla devuelve el primer valor, en caso contrario el segundo.

def clasificar_grupos_edad(edad):
    if 18 <= edad <= 24:
        return '18 a 24'
    elif 24 < edad <= 30:
        return '24 a 30'
    elif 30 < edad <= 40:
        return '30 a 40'
    elif 40 < edad <= 50:
        return '40 a 50'
    elif edad >= 60:
        return '60 en adelante'
    else:
        return 'Excluido'
     
# Aseguramos que la variable de edad (P6040) sea de tipo numérico en la tabla maestra

cgse['P6040'] = pd.to_numeric(cgse['P6040'], errors='coerce')

# Aplicamos la función fila por fila para crear la nueva columna de segmentación

cgse['Grupo_Edad_P3'] = cgse['P6040'].apply(clasificar_grupos_edad)

# Filtramos la tabla cgse para quedarnos únicamente con los grupos de interés y con las personas que pertenecen a la Fuerza de Trabajo / Población Económicamente Activa (FT == 1).

df_filtrado_p3 = cgse[(cgse['Grupo_Edad_P3'] != 'Excluido') & (cgse['FT'] == 1)].copy()


# Agrupamos de forma por 'DPTO' y 'Grupo_Edad_P3' para sumar las variables expandidas que calculamos anteriormente en el punto 2 (Totales poblacionales reales de desocupados y PEA).

agrupado_p3 = df_filtrado_p3.groupby(['DPTO', 'Grupo_Edad_P3'])[['Desocupados_Exp', 'PEA_Exp']].sum().reset_index()


# Calculamos la Tasa de Desempleo Específica (%) para cada grupo de edad dentro de cada departamento.

agrupado_p3['TD_Grupo'] = (agrupado_p3['Desocupados_Exp'] / agrupado_p3['PEA_Exp']) * 100

# Identificamos los registros con la mayor y menor tasa de desempleo para cada departamento. Las funciones idxmax() e idxmin() encuentran las posiciones de los extremos tras agrupar por departamento.

idx_max_p3 = agrupado_p3.groupby('DPTO')['TD_Grupo'].idxmax()
idx_min_p3 = agrupado_p3.groupby('DPTO')['TD_Grupo'].idxmin()

# Extraemos las filas completas correspondientes a los índices calculados.

resultado_mayor_td = agrupado_p3.loc[idx_max_p3, ['DPTO', 'Grupo_Edad_P3', 'TD_Grupo']].rename(
    columns={'Grupo_Edad_P3': 'Grupo_Mayor_TD', 'TD_Grupo': 'Mayor_TD(%)'}
)

resultado_menor_td = agrupado_p3.loc[idx_min_p3, ['DPTO', 'Grupo_Edad_P3', 'TD_Grupo']].rename(
    columns={'Grupo_Edad_P3': 'Grupo_Menor_TD', 'TD_Grupo': 'Menor_TD(%)'}
)


# Consolidamos los resultados en una única tabla unificada por departamento

tabla_final_p3 = pd.merge(resultado_mayor_td, resultado_menor_td, on='DPTO')
tabla_final_p3['DPTO'] = tabla_final_p3['DPTO'].astype(float).astype(int).astype(str).str.zfill(2)
tabla_final_p3['Nombre_Departamento'] = tabla_final_p3['DPTO'].map(dic_dptos)

# Reorganizamos las columnas para la presentación final
tabla_estructurada_p3 = tabla_final_p3[['DPTO', 'Nombre_Departamento', 'Grupo_Mayor_TD', 'Mayor_TD(%)', 'Grupo_Menor_TD', 'Menor_TD(%)']].copy()

# Desplegamos la tabla resultante en el notebook

display(tabla_estructurada_p3)

In [0]:
# En casi todos los departamentos (Antioquia, Atlántico, Bogotá D.C., Bolívar, Caldas, etc.), el grupo de edad con la mayor tasa de desempleo es el grupo de 18 a 24 años. Esto demuestra que la población más joven es la que enfrenta las barreras de entrada más altas en el mercado laboral colombiano a junio de 2026.

In [0]:
# Solución al punto cuatro del parcial

In [0]:
# Nos aseguramos de que las variables 'P6040' (edad) y 'P3271' (sexo) sean de tipo numérico

cgse['P6040'] = pd.to_numeric(cgse['P6040'], errors='coerce')  
cgse['P3271'] = pd.to_numeric(cgse['P3271'], errors='coerce')  

# Aplicamos los filtros específicos: Mujeres (2), entre 18 y 25 años, que pertenezcan a la PEA (FT == 1)

df_mujeres_jovenes = cgse[
    (cgse['P3271'] == 2) & 
    (cgse['P6040'] >= 18) & 
    (cgse['P6040'] <= 25) & 
    (cgse['FT'] == 1)
].copy()

# Agrupamos por departamento para sumar los totales poblacionales expandidos

agrupado_p4 = df_mujeres_jovenes.groupby('DPTO')[['Desocupados_Exp', 'PEA_Exp']].sum().reset_index()

# Calculamos la Tasa de Desempleo específica para este segmento de la población

agrupado_p4['TD_Mujeres_Jovenes'] = (agrupado_p4['Desocupados_Exp'] / agrupado_p4['PEA_Exp']) * 100
agrupado_p4['DPTO'] = agrupado_p4['DPTO'].astype(float).astype(int).astype(str).str.zfill(2)
agrupado_p4['Nombre_Departamento'] = agrupado_p4['DPTO'].map(dic_dptos)

# Ordenamos la tabla de menor a mayor tasa de desempleo y extraemos los 3 primeros registros (el Top 3 mínimo)

top3_menor_desempleo = agrupado_p4.sort_values(by='TD_Mujeres_Jovenes', ascending=True).head(3).copy()

# Reorganizamos las columnas 

tabla_estructurada_p4 = top3_menor_desempleo[['DPTO', 'Nombre_Departamento', 'TD_Mujeres_Jovenes']].copy()
display(tabla_estructurada_p4)

In [0]:
#El departamento del Vaupés presenta una tasa de desempleo del 0% para este segmento debido a una particularidad metodológica y estructural de la GEIH en las regiones periféricas del país. Al cruzar múltiples filtros específicos (ubicación, género, rango de edad exacto y pertenencia a la PEA), el tamaño de la muestra efectiva se reduce drásticamente, lo que puede provocar que no se registren casos de desocupación activa dentro de la muestra capturada. Asimismo, este fenómeno refleja la alta tasa de informalidad laboral y el desestimulo en la búsqueda activa de empleo en la región; muchas mujeres jóvenes se dedican al trabajo informal de subsistencia (computando como ocupadas) o dejan de buscar empleo ante la falta de mercado formal (pasando a ser económicamente inactivas), lo que matemáticamente puede producir estos resultados."

In [0]:
# Solución al punto cinco del parcial

In [0]:
# Aseguramos que las variables clave sean del tipo de dato correcto

cgse['DPTO'] = cgse['DPTO'].astype(float).astype(int).astype(str).str.zfill(2)
cgse['INGLABO'] = pd.to_numeric(cgse['INGLABO'], errors='coerce')
cgse['FEX_C18'] = pd.to_numeric(cgse['FEX_C18'], errors='coerce')

# Definimos la lista con los códigos oficiales de los departamentos solicitados

deptos_p5 = ['05', '17', '63', '66', '76']

# Filtramos: que correspondan a los departamentos de interés y que tengan ingresos válidos (> 0)

df_ingresos = cgse[(cgse['DPTO'].isin(deptos_p5)) & (cgse['INGLABO'] > 0)].copy()

# Creamos la columna del ingreso expandido (Ingreso * Factor de Expansión)

df_ingresos['Ingreso_Expandido'] = df_ingresos['INGLABO'] * df_ingresos['FEX_C18']

# 5. Agrupamos por departamento para calcular las sumatorias necesarias para el promedio ponderado
agrupado_p5 = df_ingresos.groupby('DPTO').agg(
    total_ingresos_exp=('Ingreso_Expandido', 'sum'),
    total_poblacion_exp=('FEX_C18', 'sum')
).reset_index()

# Calculamos el promedio ponderado real: Suma(Ingreso * Factor) / Suma(Factor)

agrupado_p5['Promedio_INGLABO'] = agrupado_p5['total_ingresos_exp'] / agrupado_p5['total_poblacion_exp']

# Asignamos los nombres de los departamentos 

agrupado_p5['Nombre_Departamento'] = agrupado_p5['DPTO'].map(dic_dptos)
tabla_estructurada_p5 = agrupado_p5[['DPTO', 'Nombre_Departamento', 'Promedio_INGLABO']].copy()
tabla_estructurada_p5['Promedio_INGLABO'] = tabla_estructurada_p5['Promedio_INGLABO'].round(2)
display(tabla_estructurada_p5)

In [0]:
# Los resultados muestran que el promedio de ingresos laborales para cada una de las regiones evaluadas se distribuye de la siguiente manera: Antioquia $2.486.207, Caldas $2.226.450, Quindío $2.171.559, Risaralda $2.234.061 
# y Valle del Cauca $2.147.995